# Fase 1 — ETL: Limpieza, Feature Engineering y Carga a SQL Server

Pasos 1.2, 1.3 y 1.4 del proyecto.

## Paso 1.2 — Limpieza Inicial

In [ ]:
import pandas as pd
import numpy as np

RUTA_ARCHIVO = '../../data/raw/mdi_personasdesaparecidas_pm_2017_2025.xlsx'

df = pd.read_excel(RUTA_ARCHIVO)
print(df.shape)
df.head()

In [ ]:
# Revisión de nulos
df.isnull().sum()

In [ ]:
# Estandarización de columnas de texto (provincia, canton, etc.)
columnas_texto = df.select_dtypes(include='object').columns

for col in columnas_texto:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.normalize('NFKD')
        .str.encode('ascii', errors='ignore')
        .str.decode('utf-8')
    )

df.head()

In [ ]:
# Tratamiento de nulos (ajustar según columnas reales del dataset)
# Ejemplo: df['provincia'] = df['provincia'].fillna('NO REGISTRA')
df = df.dropna(subset=['fecha_desaparicion'])  # ajustar nombre real de columna

## Paso 1.3 — Feature Engineering: `dias_solucion`

In [ ]:
df['fecha_desaparicion'] = pd.to_datetime(df['fecha_desaparicion'], errors='coerce')
df['fecha_localizacion'] = pd.to_datetime(df['fecha_localizacion'], errors='coerce')

df['dias_solucion'] = (df['fecha_localizacion'] - df['fecha_desaparicion']).dt.days

# Solo aplica para casos resueltos (localizados)
df.loc[df['fecha_localizacion'].isna(), 'dias_solucion'] = np.nan

df[['fecha_desaparicion', 'fecha_localizacion', 'dias_solucion']].head()

In [ ]:
# Guardar dataset limpio
df.to_csv('../../data/processed/casos_limpios.csv', index=False)
print('Archivo procesado guardado.')

## Paso 1.4 — Carga de Datos a SQL Server

In [ ]:
from sqlalchemy import create_engine
import urllib
import os
from dotenv import load_dotenv

load_dotenv()

SERVER = os.getenv('SQL_SERVER', 'localhost')
DATABASE = 'DB_PersonasDesaparecidas'
DRIVER = 'ODBC Driver 17 for SQL Server'

params = urllib.parse.quote_plus(
    f'DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'
)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')
print('Conexión a SQL Server lista (engine creado).')

In [ ]:
# 1) Cargar catálogos a las dimensiones primero
dim_geografia = df[['provincia', 'canton']].drop_duplicates().reset_index(drop=True)
dim_geografia.to_sql('Dim_Geografia', engine, if_exists='append', index=False)

# Repetir patrón análogo para Dim_Persona y Dim_Motivo según columnas reales del dataset

print('Dimensiones cargadas.')

In [ ]:
# 2) Recuperar llaves foráneas desde SQL y construir Fact_Casos
geo_ids = pd.read_sql('SELECT id_geografia, provincia, canton FROM Dim_Geografia', engine)

df_fact = df.merge(geo_ids, on=['provincia', 'canton'], how='left')

columnas_fact = [
    'id_geografia', 'fecha_desaparicion', 'fecha_localizacion',
    'dias_solucion', 'estado_desaparecido'
]
# Ajustar columnas_fact según el esquema final (incluir id_persona, id_motivo)

df_fact[columnas_fact].to_sql('Fact_Casos', engine, if_exists='append', index=False)
print('Tabla de hechos cargada.')